# Pokemon Type Classification — DL Assignment 1

**Goal:** Classify Pokemon images into 9 types using MLP → CNN → Transfer Learning.  
**Metric:** Macro-averaged F1 (chosen because of class imbalance — 2.76x ratio between majority and minority class).  
**Data:** 3600 train / 900 test, 64×64 PNG images, 9 classes.  
**Tasks:** Task 1 — MLP baseline | Task 2 — Custom CNN | Task 3 — EfficientNet Transfer Learning.

---
*Run this notebook top to bottom. Each section calls the tested `src/` Python files and displays results inline.*  
*All logic lives in `src/` — this notebook is a runner + display layer only.*

## How to run this notebook

**Locally** — just run all cells. `src/` and `data/` are already on disk.

**Colab** — two setup cells to run once per session:
- **Cell 3 (Setup):** clones the repo, installs dependencies.
- **Cell 4 (Data):** prompts for your `kaggle.json` → downloads the dataset automatically.
  Get your key: kaggle.com → Account → API → **Create New Token**.

After those two cells, everything else runs identically locally and on Colab.


In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────────
# Locally:  src/ is already on disk — the chdir and sys.path lines are harmless.
# Colab:    clones the repo, moves into assignment_1/, adds it to Python path,
#           installs dependencies. Run once per session.
import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("/content/DL_Proj"):
        !git clone https://github.com/fmssilva/DL_Proj.git /content/DL_Proj
    else:
        !git -C /content/DL_Proj pull --ff-only
    os.chdir("/content/DL_Proj/assignment_1")
    !pip install -r requirements.txt -q

ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# purge stale cached modules so re-runs always use the latest code on disk
stale = [k for k in sys.modules if k == "src" or k.startswith("src.")]
for k in stale:
    del sys.modules[k]
if stale:
    print(f"Cleared {len(stale)} stale src.* modules: {stale}")

# ── Shared imports ────────────────────────────────────────────────────────────
import torch
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from src.config import (BATCH_SIZE, CLASSES, DATA_DIR, EPOCHS, IMG_SIZE_SMALL,
                        LR, NUM_WORKERS, OUT_DIR, PATIENCE, SEED,
                        create_output_dirs, set_seed)

set_seed(SEED)
create_output_dirs()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}  |  PyTorch: {torch.__version__}")

CSV_PATH  = DATA_DIR / "train_labels.csv"
TRAIN_DIR = DATA_DIR / "Train"
TEST_DIR  = DATA_DIR / "Test"
print("Ready:", os.getcwd())


In [ ]:
# ── Cell 2: Data ─────────────────────────────────────────────────────────────
# Locally:  data/ already on disk — nothing to do.
# Colab:    upload your kaggle.json when prompted, then data is downloaded.
#           Get kaggle.json: kaggle.com → Account → API → Create New Token
import os, shutil

if not os.path.exists("data/train_labels.csv"):
    from google.colab import files
    files.upload()   # select kaggle.json when prompted

    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    os.replace("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

    !pip install kagglehub -q
    import kagglehub
    path = kagglehub.competition_download("the-pokemon-are-out-there-2026-task-1")
    shutil.copytree(path, "data", dirs_exist_ok=True)
    print("data/ ready:", os.listdir("data"))
else:
    print("data/ already present — skipping download.")

# ── Load CSV ──────────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print(f"Loaded CSV: {len(df)} rows, {df['label'].nunique()} classes")


---
## Part 1 — Exploratory Data Analysis

Before training anything, we look at the data to understand:
- How many samples per class (imbalance ratio)
- Whether all images are the same size (justifies the resize choice)
- What the images actually look like per class
- Per-channel pixel statistics (justifies or challenges ImageNet normalisation)

All EDA runs on the full 3600-image training set.

In [ ]:
import src.datasets.eda as eda

print("=== Class Distribution ===")
eda.class_distribution(df)

print("\n=== Image Size Distribution ===")
eda.image_size_distribution(TRAIN_DIR)

print("\n=== Data Integrity Check ===")
valid, invalid = eda.check_data_integrity(TRAIN_DIR, df)
print(f"Result: {valid} valid, {invalid} invalid")


### Plot 1 — Class Distribution

Horizontal bar chart showing the number of samples per class, sorted from most to least frequent.

In [ ]:
import src.datasets.eda_plots as eda_plots

fig = eda_plots.plot_class_distribution(df)
plt.show()
plt.close(fig)


> **Finding:** _TODO — fill in after running: note the imbalance ratio, which classes are majority/minority, and what this means for the loss function choice._

### Plot 2 — Sample Images per Class

4 random images per class (fixed seed — same grid on every run).

In [ ]:
fig = eda_plots.plot_sample_images(TRAIN_DIR, df, n_per_class=4)
plt.show()
plt.close(fig)

> **Finding:** _TODO — note visually similar class pairs (e.g. Bug/Grass both greenish, Fighting/Normal both humanoid). These are the pairs most likely to be confused by the model._

### Plot 3 — Average Image per Class

Mean pixel value across all images in each class. A blurry mean image means high intra-class variance — the class has diverse-looking images, which makes it harder to classify.

In [ ]:
fig = eda_plots.plot_average_image_per_class(TRAIN_DIR, df)
plt.show()
plt.close(fig)

> **Finding:** _TODO — fill in after running_

### Plot 4 — Per-Channel Pixel Statistics

Bar chart of per-channel mean and standard deviation (R, G, B) computed across the training set.
Also prints numeric values to compare against ImageNet normalization constants.

In [ ]:
fig = eda_plots.plot_pixel_statistics(TRAIN_DIR, df)
plt.show()
plt.close(fig)

> **Finding:** _TODO — fill in after running_

### Plot 5 — Pixel Intensity Histogram

Histogram of pixel intensity values across R, G, B channels sampled from the training set.
Reveals the overall brightness distribution and confirms whether data is approximately normalized.

In [ ]:
fig = eda_plots.plot_pixel_intensity_histogram(TRAIN_DIR, df, n_samples=200)
plt.show()
plt.close(fig)

> **Finding:** _TODO — fill in after running_

---
## Task 1 — MLP Baseline

A fully-connected Multi-Layer Perceptron treating each 64x64 RGB image as a flat vector of 12,288 features.

**Architecture:** `Flatten -> Linear(12288, 512) -> BN -> ReLU -> Dropout(0.4) -> Linear(512, 256) -> BN -> ReLU -> Dropout(0.4) -> Linear(256, 128) -> BN -> ReLU -> Dropout(0.4) -> Linear(128, 9)`

Total parameters: ~6.4M.

**Training strategy:**
- Loss: CrossEntropyLoss with class weights (handles 2.76x class imbalance)
- Optimizer: Adam (lr=1e-3)
- LR scheduler: StepLR (step_size=5, gamma=0.5)
- Early stopping: patience=5 on validation loss, saves best checkpoint

**Before running:** check `src/config.py`.
- `FAST_RUN = True` — 4 epochs, patience 2. Use this to verify the pipeline runs end-to-end (~2 min on CPU, ~1 min on GPU).
- `FAST_RUN = False` — 30 epochs, patience 5. Use this for the real training run on Colab GPU (~20 min).

In [ ]:
# check src/config.py first — FAST_RUN=True for a quick pipeline check, False for real training
# runs the full pipeline: EDA -> train -> evaluate -> save plots + submission CSV
!python task1_mlp.py

In [ ]:
from IPython.display import Image, display
display(Image("outputs/plots/task1_history.png"))
display(Image("outputs/plots/task1_confusion.png"))

> **Finding:** _TODO — fill in after running_